# Notebook 3 — Simple Hybrid RAG: Vector Search + Database Functions

This notebook combines two techniques:
- **Vector Search** finds _item master_ records by meaning (for example, “solid state drive for a laptop”).
- **Database functions** fetch exact _inventory_ and _supplier_ facts.
- The _LLM_ explains the combined results.

Run Notebook 1 first. Notebook 2 explains the SQL functions used here.
Vector Search endpoint creation requires workspace permissions and may incur cost.


In [0]:
%pip install databricks-vectorsearch databricks-langchain langchain==0.3.27
%restart_python


In [0]:
catalog_name = "main"  # Change to a catalog you can use
schema_name = "scm_e2e_rag_lab"
item_table = f"{catalog_name}.{schema_name}.item_master"
inventory_table = f"{catalog_name}.{schema_name}.inventory"
supplier_table = f"{catalog_name}.{schema_name}.supplier"
source_table = f"{catalog_name}.{schema_name}.item_master_search_text"
index_name = f"{catalog_name}.{schema_name}.item_master_vector_index"
endpoint_name = "vs_endpoint_1"
embedding_endpoint_name = "databricks-qwen3-embedding-0-6b"


## 1. Prepare item descriptions for Vector Search

We create one searchable text document per item. Only item master data is indexed—not live inventory.

In [0]:


from pyspark.sql import functions as F
search_df = spark.sql(f"""
SELECT item_id AS id,
       CONCAT('Item ID: ', item_id,
              '\nItem name: ', item_name,
              '\nCategory: ', category,
              '\nDescription: ', description) AS document
FROM {item_table}
""")
search_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(source_table)
spark.sql(f"ALTER TABLE {source_table} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
display(spark.table(source_table))


## 2. Create a Vector Search endpoint and index

Run once. If the endpoint/index already exists, inspect it in the Databricks UI rather than repeatedly creating it.

In [0]:
from databricks.vector_search.client import VectorSearchClient
vsc = VectorSearchClient(disable_notice=True)

# Create the endpoint if it does not already exist:
try:
    vsc.get_endpoint(endpoint_name)
    print(f"Endpoint '{endpoint_name}' already exists, skipping creation.")
except Exception:
    vsc.create_endpoint(name=endpoint_name, endpoint_type="STANDARD")

# Create the index after the endpoint is ready:
try:
    vsc.get_index(endpoint_name, index_name)
    print(f"Index '{index_name}' already exists, skipping creation.")
except Exception:
    vsc.create_delta_sync_index(
         endpoint_name=endpoint_name,
         index_name=index_name,
         source_table_name=source_table,
         pipeline_type="TRIGGERED",
         primary_key="id",
         embedding_source_column="document",
         embedding_model_endpoint_name=embedding_endpoint_name,
    )


## 3. Test semantic retrieval

After creating the index, wait until it is ONLINE and run this cell.

### Example question: “solid state drive for a laptop”


In [0]:
index = vsc.get_index(endpoint_name, index_name)

# Wait for the index to be ONLINE before querying
import time
for _ in range(20):
    status = index.describe().get("status", {})
    if status.get("ready", False):
        break
    print(f"Index not ready (state: {status.get('detailed_state', 'unknown')}). Waiting 15s...")
    time.sleep(15)

results = index.similarity_search(query_text="solid state drive for a laptop", columns=["document"], num_results=3)
print(results)


## 4. Create database functions for exact data

- These queries fetch current rows from the Delta tables. 
- The item descriptions come from Vector Search;stock and supplier facts (including supplier city and country) come from SQL.
- A supplier location is not the same as a warehouse location.

In [0]:


def get_inventory(item_id: str, warehouse: str):
    query = f"""
        SELECT item_id, warehouse, available_quantity
        FROM {inventory_table}
        WHERE item_id = :item_id AND warehouse = :warehouse
    """
    return spark.sql(query, args={"item_id": item_id, "warehouse": warehouse}).collect()
def get_supplier(item_id: str):
    query = f"""
        SELECT item_id, supplier_name, supplier_city, supplier_country, lead_time_days
        FROM {supplier_table}
        WHERE item_id = :item_id
    """
    return spark.sql(query, args={"item_id": item_id}).collect()
print("SSD inventory:", get_inventory("SSD001", "Chennai"))
print("SSD supplier:", get_supplier("SSD001"))  # ABC Components, Chennai, India


## 5. Combine results in a simple RAG answer

- This first version explicitly calls the functions in Python. 
- The LLM does not choose tools yet.This is easier to learn and debug. 
- The Vector Search retrieval below is enabled after the index is online.

In [0]:
# Uncomment after the Vector Search index is ONLINE and the libraries are installed.

import mlflow
from mlflow.entities import SpanType

# Enable MLflow tracing for the LangChain chain (LLM calls, prompts, retriever):
mlflow.langchain.autolog()

from databricks_langchain import ChatDatabricks, DatabricksVectorSearch
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from operator import itemgetter

vectorstore = DatabricksVectorSearch(index_name)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
model = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", temperature=0.2, max_tokens=400)

# Traced wrappers around the database functions so their calls appear as spans in the trace:
@mlflow.trace(span_type=SpanType.TOOL)
def get_inventory_traced(item_id: str, warehouse: str):
    return get_inventory(item_id, warehouse)

@mlflow.trace(span_type=SpanType.TOOL)
def get_supplier_traced(item_id: str):
    return get_supplier(item_id)

prompt = PromptTemplate.from_template("""
 You are a supply-chain assistant for a fictional computer manufacturer.
 Use the retrieved item records and the database results below.
 Do not invent facts. If stock or supplier information is missing, say so.
 Similar items are not automatically compatible substitutes.
 Retrieved item records:
 {item_context}
 Inventory query result:
 {inventory_context}
 Supplier query result (including supplier location):
 {supplier_context}
 User question: {question}
 Answer in simple language.
""")


@mlflow.trace(span_type=SpanType.CHAIN)
def answer_question(question: str, item_id: str, warehouse: str):
     docs = retriever.invoke(question)
     item_context = "\n\n".join(doc.page_content for doc in docs)
     inventory_context = str(get_inventory_traced(item_id, warehouse))
     supplier_context = str(get_supplier_traced(item_id))
     chain = prompt | model | StrOutputParser()
     return chain.invoke({
         "question": question,
         "item_context": item_context,
         "inventory_context": inventory_context,
         "supplier_context": supplier_context,
     })

print(answer_question(
     "Find the solid state drive and tell me its stock in Chennai and supplier lead time.",
     item_id="SSD001",
     warehouse="Chennai",
 ))


In [0]:

# Uncomment after the Vector Search index is ONLINE and the libraries are installed.

from databricks_langchain import ChatDatabricks, DatabricksVectorSearch
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from operator import itemgetter

vectorstore = DatabricksVectorSearch(index_name)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
model = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", temperature=0.2, max_tokens=400)

prompt = PromptTemplate.from_template("""
 You are a supply-chain assistant for a fictional computer manufacturer.
 Use the retrieved item records and the database results below.
 Do not invent facts. If stock or supplier information is missing, say so.
 Similar items are not automatically compatible substitutes.
 Retrieved item records:
 {item_context}
 Inventory query result:
 {inventory_context}
 Supplier query result (including supplier location):
 {supplier_context}
 User question: {question}
 Answer in simple language.
""")


def answer_question(question: str, item_id: str, warehouse: str):
     docs = retriever.invoke(question)
     item_context = "\n\n".join(doc.page_content for doc in docs)
     inventory_context = str(get_inventory(item_id, warehouse))
     supplier_context = str(get_supplier(item_id))
     chain = prompt | model | StrOutputParser()
     return chain.invoke({
         "question": question,
         "item_context": item_context,
         "inventory_context": inventory_context,
         "supplier_context": supplier_context,
     })

print(answer_question(
     "Find the solid state drive and tell me its stock in Chennai and supplier lead time.",
     item_id="SSD001",
     warehouse="Chennai",
 ))


## Checkpoint: what did we learn?

- Vector Search: “Which item sounds relevant to my question?”
- SQL function: “What is the exact quantity or supplier lead time?”
- RAG: “Use both sources to explain the answer.”
- Next step (not included yet): expose functions as LLM tools so an agent can choose which function to call.
